### Preprocessing: Cleaning Reddit Comments and Submissions

This notebook transforms raw Parquet data into analysis-ready datasets.

The goal is to:
- remove structurally invalid or empty content
- standardize timestamps
- reduce dataset width
- preserve as much original information as possible

No feature engineering or modeling is performed at this stage.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, from_unixtime, to_timestamp, to_date, concat_ws

In [2]:
spark = SparkSession.builder \
    .appName("wsb_preprocessing") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/19 20:41:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/19 20:41:26 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
SUBMISSIONS_RAW_PATH = "/project/macss/amritap1/redditproject/data/processed/submissions_raw_parquet"
SUBMISSIONS_CLEAN_PATH = "/project/macss/amritap1/redditproject/data/processed/submissions_clean_parquet"

COMMENTS_RAW_PATH = "/project/macss/amritap1/redditproject/data/processed/comments_raw_parquet"
COMMENTS_CLEAN_PATH = "/project/macss/amritap1/redditproject/data/processed/comments_clean_parquet"

### Submission preprocessing

Raw submissions are lightly cleaned to remove invalid or low-quality records.

Steps include:
- removing deleted or automated authors
- trimming text fields
- combining title and selftext into a single `text` field
- filtering empty or removed content
- converting timestamps into readable date formats

This produces a clean, analysis-ready dataset while preserving the original structure of discourse.

In [4]:
submissions = spark.read.parquet(SUBMISSIONS_RAW_PATH)

print("Raw submissions:", submissions.count())
submissions.printSchema()

Raw submissions: 1955840
root
 |-- id: string (nullable = true)
 |-- author: string (nullable = true)
 |-- created_utc: long (nullable = true)
 |-- title: string (nullable = true)
 |-- selftext: string (nullable = true)
 |-- score: long (nullable = true)
 |-- num_comments: long (nullable = true)
 |-- subreddit: string (nullable = true)
 |-- is_self: boolean (nullable = true)
 |-- link_flair_text: string (nullable = true)
 |-- url: string (nullable = true)
 |-- removed_by_category: string (nullable = true)



In [5]:
submissions_clean = submissions \
    .filter(col("id").isNotNull()) \
    .filter(col("author").isNotNull()) \
    .filter(~col("author").isin("[deleted]", "AutoModerator")) \
    .withColumn("title", trim(col("title"))) \
    .withColumn("selftext", trim(col("selftext"))) \
    .withColumn(
        "text",
        concat_ws(" ", col("title"), col("selftext"))
    ) \
    .filter(col("text").isNotNull()) \
    .filter(col("text") != "") \
    .filter(~col("text").isin("[deleted]", "[removed]")) \
    .withColumn("created_ts", to_timestamp(from_unixtime(col("created_utc")))) \
    .withColumn("date", to_date(col("created_ts")))

In [6]:
print("Clean submissions:", submissions_clean.count())
submissions_clean.select("text", "date").show(5, truncate=False)

Clean submissions: 1371852
+--------------------------------------------------------------------------------------------------------------------------------------------+----------+
|text                                                                                                                                        |date      |
+--------------------------------------------------------------------------------------------------------------------------------------------+----------+
|Cut your loses according to this meme. Corona season after season. Saw at homedepot                                                         |2020-02-29|
|The only people that die of corona virus are poor people old people and young girls, the exact people who dont buy or have stocks. [removed]|2020-02-29|
|DISCORD GROUP NEEDS RETARDS!                                                                                                                |2020-02-29|
|Boomer Virus! Collegehumor called it! Puts on re

In [7]:
submissions_clean.write.mode("overwrite").parquet(SUBMISSIONS_CLEAN_PATH)

print("Clean submissions written")

26/04/19 20:21:24 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

Clean submissions written


## Comment preprocessing

Raw comments are cleaned to remove invalid, low-quality, and non-informative records while preserving the core conversational structure.

Steps include:
- removing null or missing identifiers and authors
- filtering out deleted users and automated accounts (e.g., AutoModerator)
- trimming and standardizing comment text
- removing empty, deleted, or removed content
- converting Unix timestamps into structured timestamp and date fields
- selecting only relevant fields for downstream analysis (id, author, text, time, metadata)

This produces a structured, analysis-ready dataset focused on meaningful user-generated discourse while maintaining temporal and relational integrity across comment threads.

### Memory-aware processing strategy

Due to the scale of the dataset, comment preprocessing is performed incrementally rather than in a single pass.

- Files are processed sequentially in small batches
- Each batch is cleaned and immediately written to disk using append mode
- This avoids excessive memory pressure and prevents Spark driver crashes
- The pipeline remains robust across large-scale data by limiting in-memory transformations

This design ensures scalability while maintaining consistency and reproducibility of the preprocessing pipeline.

In [8]:
comments = spark.read.parquet(COMMENTS_RAW_PATH)

print("Raw comments:", comments.count())
comments.printSchema()

[Stage 10:======================================================> (56 + 2) / 58]

Raw comments: 66444342
root
 |-- id: string (nullable = true)
 |-- author: string (nullable = true)
 |-- created_utc: long (nullable = true)
 |-- body: string (nullable = true)
 |-- score: long (nullable = true)
 |-- subreddit: string (nullable = true)
 |-- author_flair_text: string (nullable = true)
 |-- controversiality: long (nullable = true)
 |-- is_submitter: boolean (nullable = true)
 |-- removed_by_category: string (nullable = true)
 |-- link_id: string (nullable = true)
 |-- parent_id: string (nullable = true)



In [4]:
COMMENTS_CLEAN_PATH = "/project/macss/amritap1/redditproject/data/processed/comments_clean_parquet"

In [5]:
import os
import shutil

if os.path.exists(COMMENTS_CLEAN_PATH):
    shutil.rmtree(COMMENTS_CLEAN_PATH)
    print("Old comments_clean_parquet folder removed")
else:
    print("No existing comments_clean_parquet folder found")

Old comments_clean_parquet folder removed


In [6]:
import glob
import os

comment_files = sorted(glob.glob("/project/macss/amritap1/redditproject/data/processed/comments_raw_parquet/*.parquet"))
print("Total comment files:", len(comment_files))

Total comment files: 706


In [7]:
from pyspark.sql.types import *

comment_schema = StructType([
    StructField("id", StringType(), True),
    StructField("author", StringType(), True),
    StructField("created_utc", LongType(), True),
    StructField("body", StringType(), True),
    StructField("score", LongType(), True),
    StructField("subreddit", StringType(), True),
    StructField("author_flair_text", StringType(), True),
    StructField("controversiality", LongType(), True),
    StructField("is_submitter", BooleanType(), True),
    StructField("removed_by_category", StringType(), True),
    StructField("link_id", StringType(), True),
    StructField("parent_id", StringType(), True)
])

In [8]:
len(comment_files)

706

In [10]:
small_batch = comment_files[:100]

for i, file_path in enumerate(small_batch):
    comments_month = spark.read.schema(comment_schema).parquet(file_path)

    comments_month_clean = comments_month \
        .filter(col("id").isNotNull()) \
        .filter(col("author").isNotNull()) \
        .filter(~col("author").isin("[deleted]", "AutoModerator")) \
        .withColumn("text", trim(col("body"))) \
        .filter(col("text").isNotNull()) \
        .filter(col("text") != "") \
        .filter(~col("text").isin("[deleted]", "[removed]")) \
        .withColumn("created_ts", to_timestamp(from_unixtime(col("created_utc")))) \
        .withColumn("date", to_date(col("created_ts"))) \
        .select(
            "id",
            "author",
            "text",
            "created_ts",
            "date",
            "score",
            "subreddit",
            "parent_id",
            "link_id"
        )

    comments_month_clean.write.mode("append").parquet(COMMENTS_CLEAN_PATH)

In [11]:
small_batch = comment_files[100:300]

for i, file_path in enumerate(small_batch):
    comments_month = spark.read.schema(comment_schema).parquet(file_path)

    comments_month_clean = comments_month \
        .filter(col("id").isNotNull()) \
        .filter(col("author").isNotNull()) \
        .filter(~col("author").isin("[deleted]", "AutoModerator")) \
        .withColumn("text", trim(col("body"))) \
        .filter(col("text").isNotNull()) \
        .filter(col("text") != "") \
        .filter(~col("text").isin("[deleted]", "[removed]")) \
        .withColumn("created_ts", to_timestamp(from_unixtime(col("created_utc")))) \
        .withColumn("date", to_date(col("created_ts"))) \
        .select(
            "id",
            "author",
            "text",
            "created_ts",
            "date",
            "score",
            "subreddit",
            "parent_id",
            "link_id"
        )

    comments_month_clean.write.mode("append").parquet(COMMENTS_CLEAN_PATH)

In [12]:
small_batch = comment_files[300:500]

for i, file_path in enumerate(small_batch):
    comments_month = spark.read.schema(comment_schema).parquet(file_path)

    comments_month_clean = comments_month \
        .filter(col("id").isNotNull()) \
        .filter(col("author").isNotNull()) \
        .filter(~col("author").isin("[deleted]", "AutoModerator")) \
        .withColumn("text", trim(col("body"))) \
        .filter(col("text").isNotNull()) \
        .filter(col("text") != "") \
        .filter(~col("text").isin("[deleted]", "[removed]")) \
        .withColumn("created_ts", to_timestamp(from_unixtime(col("created_utc")))) \
        .withColumn("date", to_date(col("created_ts"))) \
        .select(
            "id",
            "author",
            "text",
            "created_ts",
            "date",
            "score",
            "subreddit",
            "parent_id",
            "link_id"
        )

    comments_month_clean.write.mode("append").parquet(COMMENTS_CLEAN_PATH)

In [13]:
small_batch = comment_files[500:]

for i, file_path in enumerate(small_batch):
    comments_month = spark.read.schema(comment_schema).parquet(file_path)

    comments_month_clean = comments_month \
        .filter(col("id").isNotNull()) \
        .filter(col("author").isNotNull()) \
        .filter(~col("author").isin("[deleted]", "AutoModerator")) \
        .withColumn("text", trim(col("body"))) \
        .filter(col("text").isNotNull()) \
        .filter(col("text") != "") \
        .filter(~col("text").isin("[deleted]", "[removed]")) \
        .withColumn("created_ts", to_timestamp(from_unixtime(col("created_utc")))) \
        .withColumn("date", to_date(col("created_ts"))) \
        .select(
            "id",
            "author",
            "text",
            "created_ts",
            "date",
            "score",
            "subreddit",
            "parent_id",
            "link_id"
        )

    comments_month_clean.write.mode("append").parquet(COMMENTS_CLEAN_PATH)